# BARAM 2026 Preprocessing Reference

This notebook is a preprocessing-only reference for the wind generation task.  
Use this notebook as the base feature engineering recipe for tree models, deep learning models, or later ablation experiments.

## 1. Imports and Configuration

In [ ]:
from itertools import combinations
from pathlib import Path
import gc
import json
import random
import time

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

COLAB_DATA_DIR = Path("/content/drive/MyDrive/BARAM2026/data")
LOCAL_DATA_DIR = Path(r"C:\Users\심현석\Documents\BARAM 2026")

if IN_COLAB:
    BASE_DIR = COLAB_DATA_DIR
elif LOCAL_DATA_DIR.exists():
    BASE_DIR = LOCAL_DATA_DIR
else:
    BASE_DIR = Path.cwd()

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "preprocess_reference"
FEATURE_OUTPUT_DIR = OUTPUT_DIR / f"{EXPERIMENT_NAME}_features"
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21_600.0,
    "kpx_group_2": 21_600.0,
    "kpx_group_3": 21_000.0,
}

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

WEATHER_AGGS = ["mean", "std", "min", "max"]
SUBSET_AGGS = ["mean", "std", "min", "max"]

# Convert LDAPS precipitation into one-hot style binary flags.
# The 15mm+ bin is the baseline bin, so it is intentionally not made as a feature.
PRECIP_SOURCE_COL = "surface_0_ncpcp"
PRECIP_BIN_FEATURES = [
    ("precip_0_1mm", 0.0, 1.0),
    ("precip_1_3mm", 1.0, 3.0),
    ("precip_3_15mm", 3.0, 15.0),
]

# Choose whether flag columns are counted during spatial aggregation.
COUNT_REGIME_FLAGS_IN_AGGREGATIONS = False
COUNT_PRECIP_FLAGS_IN_AGGREGATIONS = False

# Choose whether to aggregate every grid from each weather source.
INCLUDE_LDAPS_ALL_GRID_AGGREGATIONS = True
INCLUDE_GFS_ALL_GRID_AGGREGATIONS = True

# Choose the LDAPS raw grids used directly for each target group.
LDAPS_GROUP_RAW_GRID_IDS = {
    "kpx_group_1": [5, 6],
    "kpx_group_2": [6, 11],
    "kpx_group_3": [6, 12],
}

# Choose the LDAPS grid groups used for group-specific aggregation.
LDAPS_GROUP_AGGREGATION_REFERENCE_GRIDS = {
    "kpx_group_1": [[5, 6, 10, 11]],
    "kpx_group_2": [[6, 7, 11, 12]],
    "kpx_group_3": [[6, 11, 12, 16]],
}

# Choose which aggregation combination sizes to make from the reference grids.
# The default [4] means 4C4 only for each 4-grid reference group.
LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES = [4]

# Optional RAM guard. Set this to an integer, such as 500, if lag/rolling features are still too many.
# MAX_LAG_ROLL_BASE_COLS = 500
WEATHER_LAGS = [1, 2, 24]
WEATHER_ROLL_WINDOWS = [3, 6, 24]

print("BASE_DIR:", BASE_DIR)
print("FEATURE_OUTPUT_DIR:", FEATURE_OUTPUT_DIR)

## 2. Load Data

- 데이터 불러오기
- train_labels에서 2022년 그룹 3의 발전량은 NaN으로 두기

In [2]:
required_paths = [
    TRAIN_DIR / "train_labels.csv",
    TRAIN_DIR / "ldaps_train.csv",
    TRAIN_DIR / "gfs_train.csv",
    TEST_DIR / "ldaps_test.csv",
    TEST_DIR / "gfs_test.csv",
    BASE_DIR / "sample_submission.csv",
]

labels_raw = pd.read_csv(TRAIN_DIR / "train_labels.csv")
labels_raw["kst_dtm"] = pd.to_datetime(labels_raw["kst_dtm"])
labels_raw = labels_raw.sort_values("kst_dtm").reset_index(drop=True)

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv")
sample_submission = pd.read_csv(BASE_DIR / "sample_submission.csv")

for df in [ldaps_train, gfs_train, ldaps_test, gfs_test]:
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    df["data_available_kst_dtm"] = pd.to_datetime(df["data_available_kst_dtm"])

sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("labels:", labels_raw.shape)
print("ldaps_train/test:", ldaps_train.shape, ldaps_test.shape)
print("gfs_train/test:", gfs_train.shape, gfs_test.shape)
print("sample_submission:", sample_submission.shape)

labels: (26304, 4)
ldaps_train/test: (420864, 35) (140160, 35)
gfs_train/test: (236736, 40) (78840, 40)
sample_submission: (8760, 5)


In [3]:
def count_missing_values(labels: pd.DataFrame) -> pd.DataFrame:
    out = labels.sort_values("kst_dtm").copy()
    out["kst_dtm"] = pd.to_datetime(out["kst_dtm"])

    cutoff = pd.Timestamp("2023-01-01 00:00:00")
    audit_rows = []

    for target in TARGET_COLS:
        missing_mask = out[target].isna()
        if target == "kpx_group_3":
            count = int((missing_mask & (out["kst_dtm"] > cutoff)).sum())
        else:
            count = int(missing_mask.sum())
        audit_rows.append({"kpx_group": target, "count": count})

    return pd.DataFrame(audit_rows)

display(count_missing_values(labels_raw))

,kpx_group,count
0,kpx_group_1,104
1,kpx_group_2,103
2,kpx_group_3,6


그룹 3의 발전량 데이터의 결측값 확인

In [4]:
labels_raw[labels_raw['kpx_group_3'].isna() & (labels_raw["kst_dtm"] > pd.Timestamp("2023-01-01 00:00:00"))]

,kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
22139,2024-07-11 12:00:00,NaN,NaN,NaN
22140,2024-07-11 13:00:00,NaN,NaN,NaN
22141,2024-07-11 14:00:00,NaN,NaN,NaN
22595,2024-07-30 12:00:00,NaN,NaN,NaN
22596,2024-07-30 13:00:00,NaN,NaN,NaN
22597,2024-07-30 14:00:00,NaN,NaN,NaN


## 3. Drop Columns
- 일부 기상변수 제거

In [ ]:
LDAPS_DROP_COLS = [
    "etc_0_blh",
    "surface_0_NDNSW",
    "surface_0_NDNLW",
    "heightAboveGround_2_SWDIR",
    "heightAboveGround_2_SWDIF",
    "etc_0_hcc",
    "etc_0_mcc",
    "etc_0_lcc",
    "etc_0_VLCDC",
    "surface_0_avg_lsprate",
    "surface_0_lssrate",
    "meanSea_0_prmsl",
    # "surface_0_ncpcp",
    "surface_0_snol",
    "surface_0_SNOM",
    "surface_0_lsm",
    "surface_0_h",
]

GFS_DROP_COLS = [
    "planetaryBoundaryLayer_0_u",
    "planetaryBoundaryLayer_0_v",
    "planetaryBoundaryLayer_0_VRATE",
    "meanSea_0_prmsl",
    "surface_0_dswrf",
    "surface_0_dlwrf",
    "surface_0_prate",
    "surface_0_tp",
    "lowCloudLayer_0_lcc",
    "middleCloudLayer_0_mcc",
    "highCloudLayer_0_hcc",
    "atmosphere_0_tcc",
]

def drop_existing_columns(df: pd.DataFrame, cols: list[str], name: str) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    if existing:
        df = df.drop(columns=existing)
    print(f"{name}: dropped {len(existing)} columns | remaining columns={df.shape[1]}")
    return df

ldaps_train = drop_existing_columns(ldaps_train, LDAPS_DROP_COLS, "ldaps_train")
ldaps_test = drop_existing_columns(ldaps_test, LDAPS_DROP_COLS, "ldaps_test")
gfs_train = drop_existing_columns(gfs_train, GFS_DROP_COLS, "gfs_train")
gfs_test = drop_existing_columns(gfs_test, GFS_DROP_COLS, "gfs_test")

## 3.1 Precipitation Distribution Check

Check the distribution of `surface_0_ncpcp` in `ldaps_train` using the 0~1mm, 1~3mm, 3~15mm, and 15mm+ bins, ignoring `grid_id`.


In [ ]:
precip_col = PRECIP_SOURCE_COL
precip_bins = [0, 1, 3, 15, np.inf]
precip_labels = ["0~1mm", "1~3mm", "3~15mm", "15mm~"]

ldaps_precip = ldaps_train.copy()
ldaps_precip["forecast_kst_dtm"] = pd.to_datetime(ldaps_precip["forecast_kst_dtm"])

ldaps_precip["precip_bin"] = pd.cut(
    ldaps_precip[precip_col].astype(float).clip(lower=0),
    bins=precip_bins,
    labels=precip_labels,
    right=False,
    include_lowest=True,
)

precip_train_overall = (
    ldaps_precip["precip_bin"]
    .value_counts()
    .reindex(precip_labels, fill_value=0)
    .rename_axis("precip_bin")
    .reset_index(name="count")
)
precip_train_overall["ratio"] = precip_train_overall["count"] / precip_train_overall["count"].sum()
precip_train_overall["ratio_pct"] = precip_train_overall["ratio"] * 100
display(precip_train_overall)

## 4. Physics-Feature Engineering Helpers

1. 물리 기반 파생변수 생성 함수 정의
   - 풍속(sqrt(u^2+v^2)), 풍속^2, 풍속^3, 풍속^(-2), 풍속^(-3)
   - sin(풍향), cos(풍향)
   - 공기 밀도
   - power_proxy: 0.5 * 공기 밀도 * swept area * 유효풍속^3 <br><br>

2. 그룹별로 참조하는 grid의 범위를 정의

In [ ]:
LDAPS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "wind_10m"),
    ("heightAboveGround_50_50MUmax", "heightAboveGround_50_50MVmax", "wind_50m_max"),
    ("heightAboveGround_50_50MUmin", "heightAboveGround_50_50MVmin", "wind_50m_min"),
    ("heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS", "xbl_wind_5m"),
]

GFS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "wind_10m"),
    ("heightAboveGround_80_u", "heightAboveGround_80_v", "wind_80m"),
    ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "wind_100m"),
    ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl_wind"),
    ("isobaricInhPa_850_u", "isobaricInhPa_850_v", "wind_850hpa"),
    ("isobaricInhPa_700_u", "isobaricInhPa_700_v", "wind_700hpa"),
    ("isobaricInhPa_500_u", "isobaricInhPa_500_v", "wind_500hpa"),
]

LDAPS_SPEED_COLS = [f"{name}_speed" for _, _, name in LDAPS_UV_PAIRS]
GFS_SPEED_COLS = [f"{name}_speed" for _, _, name in GFS_UV_PAIRS]

GROUP_REGIME_SPECS = {
    "kpx_group_1": {"alias": "g1", "cut_in": 3.0, "rated": 12.0, "cut_out": 22.5},
    "kpx_group_2": {"alias": "g2", "cut_in": 3.0, "rated": 12.0, "cut_out": 22.5},
    "kpx_group_3": {"alias": "g3", "cut_in": 3.0, "rated": 12.5, "cut_out": 22.0},
}


# Build a readable subset name such as g1_s_5_6_10_11.
def subset_name(alias: str, grid_ids: list[int]) -> str:
    return f"{alias}_s_{'_'.join(map(str, grid_ids))}"


# Generate aggregation subsets from the requested combination sizes.
def combination_subsets(alias: str, grid_ids: list[int], sizes: list[int] | tuple[int, ...]) -> dict[str, list[int]]:
    subsets = {}
    for size in sizes:
        if size <= 1 or size > len(grid_ids):
            continue
        for combo in combinations(grid_ids, size):
            combo_ids = list(combo)
            subsets[subset_name(alias, combo_ids)] = combo_ids
    return subsets


# Merge aggregation subsets from one or more reference grid groups.
def merge_combination_subsets(
    alias: str,
    grid_groups: list[list[int]],
    sizes: list[int] | tuple[int, ...],
) -> dict[str, list[int]]:
    subsets = {}
    for grid_ids in grid_groups:
        subsets.update(combination_subsets(alias, grid_ids, sizes))
    return subsets


# Build the final LDAPS spatial plan from the configuration cell.
def build_ldaps_spatial_plan(
    raw_grid_ids: dict[str, list[int]],
    aggregation_reference_grids: dict[str, list[list[int]]],
    combination_sizes: list[int] | tuple[int, ...],
) -> dict[str, dict[str, object]]:
    plan = {}
    for target in TARGET_COLS:
        alias = GROUP_REGIME_SPECS[target]["alias"]
        plan[target] = {
            "raw_grid_ids": raw_grid_ids.get(target, []),
            "subsets": merge_combination_subsets(
                alias,
                aggregation_reference_grids.get(target, []),
                combination_sizes,
            ),
        }
    return plan


LDAPS_GROUP_SPATIAL_PLAN = build_ldaps_spatial_plan(
    LDAPS_GROUP_RAW_GRID_IDS,
    LDAPS_GROUP_AGGREGATION_REFERENCE_GRIDS,
    LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES,
)

GFS_RAW_GRID_IDS = [5]


# Calculate air density by using weather features.
def safe_inverse(values: pd.Series, power: int = 1) -> pd.Series:
    vals = values.astype(float)
    return pd.Series(np.where(vals > 0, 1.0 / np.power(vals, power), np.nan), index=values.index)

def add_air_density_features(out: pd.DataFrame) -> pd.DataFrame:
    temp_col = next((c for c in ["heightAboveGround_2_t", "heightAboveGround_2_2t"] if c in out.columns), None)
    humidity_col = next((c for c in ["heightAboveGround_2_q", "heightAboveGround_2_2sh"] if c in out.columns), None)
    pressure_col = "surface_0_sp" if "surface_0_sp" in out.columns else None
    if temp_col is None or humidity_col is None or pressure_col is None:
        return out

    temp_k = out[temp_col].astype(float).clip(lower=150.0, upper=360.0)
    specific_humidity = out[humidity_col].astype(float).clip(lower=0.0, upper=0.08)
    pressure_pa = out[pressure_col].astype(float).clip(lower=50_000.0, upper=110_000.0)

    # 공기 밀도는 습도를 고려하는 공식으로 고정
    out["humid_air_density"] = pressure_pa / (287.05 * temp_k * (1.0 + 0.61 * specific_humidity))
    out["humid_air_density_inv"] = safe_inverse(out["humid_air_density"], power=1)
    
    return out

    
# Add physics features by using wind features
def add_wind_physics_features(df: pd.DataFrame, uv_pairs: list[tuple[str, str, str]], target: str) -> pd.DataFrame:
    out = df.copy()
    speed_cols = []
    for u_col, v_col, name in uv_pairs:
        if u_col not in out.columns or v_col not in out.columns:
            continue

        u = out[u_col].astype(float)
        v = out[v_col].astype(float)
        speed = np.sqrt(u * u + v * v)
        angle = np.arctan2(v, u)
        speed_col = f"{name}_speed"

        out[speed_col] = speed
        out[f"{name}_speed_sq"] = speed ** 2
        out[f"{name}_speed_cubed"] = speed ** 3
        out[f"{name}_angle_sin"] = np.sin(angle)
        out[f"{name}_angle_cos"] = np.cos(angle)
        speed_cols.append(speed_col)

    out = add_air_density_features(out)
    
    spec = GROUP_REGIME_SPECS[target]
    rotor_radius_m = 68.0 if target == "kpx_group_3" else 63.0
    swept_area = np.pi * rotor_radius_m * rotor_radius_m

    for speed_col in speed_cols:
        raw_speed = out[speed_col].astype(float)
        effective_speed = raw_speed.copy()
        effective_speed = effective_speed.mask(raw_speed < spec["cut_in"], 0.0)
        effective_speed = effective_speed.mask(raw_speed >= spec["cut_out"], 0.0)
        effective_speed = effective_speed.mask((raw_speed >= spec["rated"]) & (raw_speed < spec["cut_out"]), spec["rated"])

        out[f"{speed_col}_power_proxy"] = 0.5 * out["humid_air_density"] * swept_area * effective_speed ** 3
    
    return out

## 5. Regime Flag and Spatial Aggregation Helpers

1. 풍속을 구간화하는 0-1 변수 생성 함수 (regime flag)

2. 원본 데이터는 하나의 예보 대상 시각에 대해 16개 or 9개의 행이 존재하므로, 각 grid별 기상 예보값을 하나의 행에 합쳐놓는 함수

3. 각 예보 대상 시각별로, 모든/일부 grid에 대해 기상 예보값을 요약하는 파생변수 생성 함수 (mean, std, max, min)

In [ ]:
# Create binary wind-regime flags for each available wind-speed column.
def add_group_speed_regime_flags(grid_df: pd.DataFrame, target: str, speed_cols: list[str]) -> pd.DataFrame:
    out = grid_df.copy()

    # Load the cut-in, rated, and cut-out speeds for the target group.
    spec = GROUP_REGIME_SPECS[target]
    cut_in = spec["cut_in"]
    rated = spec["rated"]
    cut_out = spec["cut_out"]

    # Create one flag for each wind-speed regime.
    for speed_col in speed_cols:
        if speed_col not in out.columns:
            continue
        speed = out[speed_col].astype(float)
        out[f"{speed_col}_below_cutin"] = (speed < cut_in).astype(float)
        out[f"{speed_col}_cutin_to_rated"] = ((speed >= cut_in) & (speed < rated)).astype(float)
        out[f"{speed_col}_rated_to_cutout"] = ((speed >= rated) & (speed < cut_out)).astype(float)

    return out


# Return columns created by add_group_speed_regime_flags.
def regime_flag_cols(df: pd.DataFrame) -> list[str]:
    suffixes = ("_below_cutin", "_cutin_to_rated", "_rated_to_cutout")
    return [c for c in df.columns if c.endswith(suffixes)]


# Return precipitation-bin flag columns created by add_precipitation_bin_flags.
def precip_flag_cols(df: pd.DataFrame) -> list[str]:
    names = [name for name, _, _ in PRECIP_BIN_FEATURES]
    return [c for c in names if c in df.columns]


# Convert raw precipitation into binary bin flags. The 15mm+ bin is kept as the omitted baseline.
def add_precipitation_bin_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if PRECIP_SOURCE_COL not in out.columns:
        return out

    precip = out[PRECIP_SOURCE_COL].astype(float).clip(lower=0)
    for name, lower, upper in PRECIP_BIN_FEATURES:
        out[name] = ((precip >= lower) & (precip < upper)).astype(float)

    return out


# Select numeric weather feature columns while excluding identifiers, timestamps, locations, and raw u/v components.
def numeric_weather_cols(
    df: pd.DataFrame,
    exclude_regime_flags: bool = False,
    exclude_precip_flags: bool = False,
) -> list[str]:
    # Exclude the raw precipitation column because only the precipitation-bin flags should reach the final data.
    skip = {"forecast_id", "forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude", PRECIP_SOURCE_COL}
    uv_component_cols = {col for u_col, v_col, _ in [*LDAPS_UV_PAIRS, *GFS_UV_PAIRS] for col in (u_col, v_col)}
    skip.update(uv_component_cols)
    cols = [c for c in df.columns if c not in skip and pd.api.types.is_numeric_dtype(df[c])]

    # Optionally remove wind-regime flags from numeric aggregation bases.
    if exclude_regime_flags:
        flag_set = set(regime_flag_cols(df))
        cols = [c for c in cols if c not in flag_set]

    # Optionally remove precipitation-bin flags from numeric aggregation bases.
    if exclude_precip_flags:
        flag_set = set(precip_flag_cols(df))
        cols = [c for c in cols if c not in flag_set]

    return cols


# Pivot selected grid IDs into wide columns without aggregating them.
def raw_grid_wide(grid_df: pd.DataFrame, prefix: str, grid_ids: list[int]) -> pd.DataFrame:
    # Start from one row per forecast time.
    out = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    cols = numeric_weather_cols(grid_df, exclude_regime_flags=False, exclude_precip_flags=False)

    # Add one set of feature columns for each selected grid_id.
    for gid in grid_ids:
        sub = grid_df.loc[grid_df["grid_id"] == gid, ["forecast_kst_dtm", *cols]].copy()
        rename = {c: f"{prefix}_g{gid:02d}_{c}" for c in cols}
        out = out.merge(sub.rename(columns=rename), on="forecast_kst_dtm", how="left")

    return out


# Aggregate all available grids into one row per forecast time.
def aggregate_all_grids(grid_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    # Select base numeric columns for mean/std/min/max aggregation.
    time_index = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    base_cols = numeric_weather_cols(grid_df, exclude_regime_flags=True, exclude_precip_flags=True)
    flag_cols = regime_flag_cols(grid_df)
    precip_cols = precip_flag_cols(grid_df)

    # Store each aggregation block before merging them together.
    pieces = []

    # Aggregate ordinary numeric weather features across all grids.
    if base_cols:
        agg = grid_df.groupby("forecast_kst_dtm", sort=True)[base_cols].agg(WEATHER_AGGS)
        agg.columns = [f"{prefix}_all_{col}_{stat}" for col, stat in agg.columns]
        pieces.append(agg.reset_index())

    # Optionally count how many grids fall into each wind-speed regime.
    if COUNT_REGIME_FLAGS_IN_AGGREGATIONS and flag_cols:
        counts = grid_df.groupby("forecast_kst_dtm", sort=True)[flag_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_all_{c}_count" for c in flag_cols})
        pieces.append(counts)

    # Optionally count how many grids fall into each precipitation bin. The 15mm+ baseline is not counted.
    if COUNT_PRECIP_FLAGS_IN_AGGREGATIONS and precip_cols:
        counts = grid_df.groupby("forecast_kst_dtm", sort=True)[precip_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_all_{c}_count" for c in precip_cols})
        pieces.append(counts)

    # Merge all aggregation blocks back onto the forecast-time index.
    out = time_index
    for frame in pieces:
        out = out.merge(frame, on="forecast_kst_dtm", how="left")

    return out


# Aggregate one selected subset of grid IDs into one row per forecast time.
def aggregate_subset(grid_df: pd.DataFrame, prefix: str, subset_name: str, grid_ids: list[int]) -> pd.DataFrame:
    # Select base numeric columns for mean/std/min/max aggregation.
    time_index = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    base_cols = numeric_weather_cols(grid_df, exclude_regime_flags=True, exclude_precip_flags=True)
    flag_cols = regime_flag_cols(grid_df)
    precip_cols = precip_flag_cols(grid_df)
    selected_cols = ["forecast_kst_dtm", "grid_id", *base_cols, *flag_cols, *precip_cols]
    sub = grid_df.loc[grid_df["grid_id"].isin(grid_ids), selected_cols].copy()
    if sub.empty:
        return time_index

    # Store each aggregation block before merging them together.
    pieces = []

    # Aggregate ordinary numeric weather features across the selected grids.
    if base_cols:
        agg = sub.groupby("forecast_kst_dtm", sort=True)[base_cols].agg(SUBSET_AGGS)
        agg.columns = [f"{prefix}_{subset_name}_{col}_{stat}" for col, stat in agg.columns]
        pieces.append(agg.reset_index())

    # Optionally count how many selected grids fall into each wind-speed regime.
    if COUNT_REGIME_FLAGS_IN_AGGREGATIONS and flag_cols:
        counts = sub.groupby("forecast_kst_dtm", sort=True)[flag_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_{subset_name}_{c}_count" for c in flag_cols})
        pieces.append(counts)

    # Optionally count how many selected grids fall into each precipitation bin. The 15mm+ baseline is not counted.
    if COUNT_PRECIP_FLAGS_IN_AGGREGATIONS and precip_cols:
        counts = sub.groupby("forecast_kst_dtm", sort=True)[precip_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_{subset_name}_{c}_count" for c in precip_cols})
        pieces.append(counts)

    # Merge all aggregation blocks back onto the forecast-time index.
    out = time_index
    for frame in pieces:
        out = out.merge(frame, on="forecast_kst_dtm", how="left")

    return out

## 6. Calendar, Lag and Rolling Helpers

1. 시간 파생변수 생성 함수 정의
   - hour, day, month, season
   - 각각 sin, cos 변환
   - 2024년은 윤년임을 고려하여 변환함 <br><br>

2. lag, rolling 파생변수 생성 함수 정의

In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out["forecast_kst_dtm"])

    hour = dt.dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    day = dt.dt.dayofyear
    days_in_year = np.where(dt.dt.is_leap_year, 366, 365)
    out["day_sin"] = np.sin(2 * np.pi * day / days_in_year)
    out["day_cos"] = np.cos(2 * np.pi * day / days_in_year)

    month = dt.dt.month
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)

    season = ((dt.dt.month % 12) // 3).astype(int)
    out["season_sin"] = np.sin(2 * np.pi * season / 4)
    out["season_cos"] = np.cos(2 * np.pi * season / 4)

    return out


def choose_lag_roll_cols(df: pd.DataFrame, subset_names: list[str], max_cols: int | None = None) -> list[str]:
    # Use the global limit only when it exists and the caller did not pass a limit.
    if max_cols is None:
        max_cols = globals().get("MAX_LAG_ROLL_BASE_COLS", None)

    weather_keywords = [
        "wind", "speed", "gust", "power_proxy", "precip",
        "humid_air_density", "surface_0_sp",
        "heightAboveGround_2_t", "heightAboveGround_2_2t",
        "heightAboveGround_2_q", "heightAboveGround_2_2sh",
    ]
    regime_keywords = ["below_cutin", "cutin_to_rated", "rated_to_cutout"]

    # 이 컬럼이 raw_grid_wide()에서 나온 컬럼인지 확인
    def is_raw_grid_col(col: str) -> bool:
        parts = col.split("_")
        return len(parts) >= 3 and len(parts[1]) == 3 and parts[1].startswith("g") and parts[1][1:].isdigit()

    # 이 컬럼이 현재 그룹의 subset aggregation에서 나온 컬럼인지 확인
    def is_subset_col(col: str) -> bool:
        return any(subset_name in col for subset_name in subset_names)

    # 이미 만들어진 lag/rolling 컬럼인지 확인
    def is_existing_lag_roll_col(col: str) -> bool:
        return "_lag" in col or "_roll" in col

    # 수치형 변수만 골라내기
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    # Keep numeric weather, regime, raw-grid, and subset columns as possible lag/rolling bases.
    candidates = [
        col
        for col in numeric_cols
        if not is_existing_lag_roll_col(col)
        and (
            any(keyword in col for keyword in weather_keywords)
            or any(keyword in col for keyword in regime_keywords)
            or is_raw_grid_col(col)
            or is_subset_col(col)
        )
    ]

    # 개수 제한이 없거나, 후보 개수가 제한보다 작으면 그대로 반환
    if max_cols is None or len(candidates) <= max_cols:
        return candidates

    ##################### 이 아래는 후보가 너무 많을 때만 실행됨 #####################
    
    # subset 관련 컬럼은 무조건 포함하도록 함
    subset_cols = [col for col in candidates if is_subset_col(col)]

    # raw grid, 공기 밀도, power_proxy, regime flag 컬럼은 무조건 포함하도록 함
    protected_cols = [
        col
        for col in candidates
        if is_raw_grid_col(col)
        or "humid_air_density" in col
        or "power_proxy" in col
        or "precip" in col
        or any(keyword in col for keyword in regime_keywords)
    ]

    # subset_cols와 protected_cols를 합치되 중복은 제거
    protected = list(dict.fromkeys(subset_cols + protected_cols))

    # Return all protected columns when protected columns alone exceed the requested limit.
    if len(protected) > max_cols:
        selected = protected
        excluded = [col for col in candidates if col not in selected]
        choose_lag_roll_cols.last_excluded_cols = excluded
        print(f"[WARN] Protected columns ({len(protected)}) exceed max_cols ({max_cols}). Return all protected columns.")
        return selected
    
    # 후보 컬럼들의 분산을 계산.
    # 분산이 큰 컬럼은 시간에 따라 값이 많이 변하는 컬럼이고, lag/rolling feature를 만들었을 때 정보량이 있을 가능성이 더 크다고 보기.
    variances = df[candidates].var(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # 분산이 큰 순서대로 컬럼 정렬, 이미 보호된 컬럼은 제외. max_cols개까지만 선택.
    ranked = [col for col in variances.sort_values(ascending=False).index.tolist() if col not in protected]
    selected = (protected + ranked)[:max_cols]
    excluded = [col for col in candidates if col not in selected]
    choose_lag_roll_cols.last_excluded_cols = excluded

    return selected


def add_lag_rolling_features(df: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    out = df.sort_values("forecast_kst_dtm").reset_index(drop=True).copy()
    new_features = {}
    
    for col in base_cols:
        s = out[col].astype(float)

        # Lag 파생변수 생성 (1, 2, 3, 24)
        for lag in WEATHER_LAGS:
            new_features[f"{col}_lag{lag}"] = s.shift(lag)
        shifted = s.shift(1)

        # Rolling 파생변수 생성 (3, 6, 12, 24)
        for window in WEATHER_ROLL_WINDOWS:
            new_features[f"{col}_roll{window}_mean"] = shifted.rolling(window, min_periods=1).mean()
            new_features[f"{col}_roll{window}_std"] = shifted.rolling(window, min_periods=2).std()

    if new_features:
        out = pd.concat([out, pd.DataFrame(new_features, index=out.index)], axis=1)
        
    return out


def merge_feature_frames(frames: list[pd.DataFrame]) -> pd.DataFrame:
    non_empty = [f for f in frames if f is not None and len(f.columns) > 1]
    if not non_empty:
        raise ValueError("No feature frames to merge.")
    out = non_empty[0]
    for frame in non_empty[1:]:
        out = out.merge(frame, on="forecast_kst_dtm", how="outer")
    return out

## 7. Build Group-Specific Feature Frames

Combine calendar features, optional all-grid LDAPS/GFS aggregations, group-specific LDAPS raw grid features, and selected subset aggregation features. Raw LDAPS grids, aggregation reference grids, precipitation flag counts, all-grid aggregation toggles, and aggregation combination sizes are controlled in the configuration cell.


In [ ]:
started = time.time()

FEATURE_INVENTORY_PATH = FEATURE_OUTPUT_DIR / "feature_inventory.csv"
labels = labels_raw.copy()


def prepare_weather_grid(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.copy()
    out["forecast_kst_dtm"] = pd.to_datetime(out["forecast_kst_dtm"])
    out["data_available_kst_dtm"] = pd.to_datetime(out["data_available_kst_dtm"])
    out = out.sort_values(["forecast_kst_dtm", "grid_id", "data_available_kst_dtm"])
    out = out.groupby(["forecast_kst_dtm", "grid_id"], as_index=False).tail(1).reset_index(drop=True)
    out = add_precipitation_bin_flags(out)
    return out


def add_target_wind_physics_features(df: pd.DataFrame, uv_pairs: list[tuple[str, str, str]], target: str) -> pd.DataFrame:
    out = df.copy()
    speed_cols = []

    for u_col, v_col, name in uv_pairs:
        if u_col not in out.columns or v_col not in out.columns:
            continue
        u = out[u_col].astype(float)
        v = out[v_col].astype(float)
        speed = np.sqrt(u * u + v * v)
        angle = np.arctan2(v, u)
        speed_col = f"{name}_speed"

        out[speed_col] = speed
        out[f"{name}_speed_sq"] = speed ** 2
        out[f"{name}_speed_cubed"] = speed ** 3
        out[f"{name}_angle_sin"] = np.sin(angle)
        out[f"{name}_angle_cos"] = np.cos(angle)
        speed_cols.append(speed_col)

    temp_col = next((c for c in ["heightAboveGround_2_t", "heightAboveGround_2_2t"] if c in out.columns), None)
    humidity_col = next((c for c in ["heightAboveGround_2_q", "heightAboveGround_2_2sh"] if c in out.columns), None)
    pressure_col = "surface_0_sp" if "surface_0_sp" in out.columns else None

    if temp_col is not None and humidity_col is not None and pressure_col is not None:
        temp_k = out[temp_col].astype(float).clip(lower=150.0, upper=360.0)
        specific_humidity = out[humidity_col].astype(float).clip(lower=0.0, upper=0.08)
        pressure_pa = out[pressure_col].astype(float).clip(lower=50_000.0, upper=110_000.0)
        out["humid_air_density"] = pressure_pa / (287.05 * temp_k * (1.0 + 0.61 * specific_humidity))
        out["humid_air_density_inv"] = safe_inverse(out["humid_air_density"], power=1)

        spec = GROUP_REGIME_SPECS[target]
        rotor_radius_m = 68.0 if target == "kpx_group_3" else 63.0
        swept_area = np.pi * rotor_radius_m * rotor_radius_m

        for speed_col in speed_cols:
            raw_speed = out[speed_col].astype(float)
            effective_speed = raw_speed.copy()
            effective_speed = effective_speed.mask(raw_speed < spec["cut_in"], 0.0)
            effective_speed = effective_speed.mask(raw_speed >= spec["cut_out"], 0.0)
            effective_speed = effective_speed.mask((raw_speed >= spec["rated"]) & (raw_speed < spec["cut_out"]), spec["rated"])
            out[f"{speed_col}_power_proxy"] = 0.5 * out["humid_air_density"] * swept_area * effective_speed ** 3

    return out


ldaps_all = pd.concat([ldaps_train, ldaps_test], ignore_index=True)
gfs_all = pd.concat([gfs_train, gfs_test], ignore_index=True)

print("Preparing LDAPS grid-level data...")
ldaps_grid = prepare_weather_grid(ldaps_all, "ldaps")
print("Preparing GFS grid-level data...")
gfs_grid = prepare_weather_grid(gfs_all, "gfs")

ldaps_locations = ldaps_grid[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)
gfs_locations = gfs_grid[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)
display(ldaps_locations)
display(gfs_locations)

calendar_features = add_calendar_features(
    pd.concat([
        ldaps_grid[["forecast_kst_dtm"]],
        gfs_grid[["forecast_kst_dtm"]],
    ], ignore_index=True).drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
)

group_feature_sets = {}
feature_inventory_rows = []

print("LDAPS aggregation combination sizes:", LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES)

for target in TARGET_COLS:
    print(f"\nBuilding group-specific features for {target}")
    plan = LDAPS_GROUP_SPATIAL_PLAN[target]
    raw_ids = plan["raw_grid_ids"]
    subset_names = list(plan["subsets"].keys())

    ldaps_group_grid = add_target_wind_physics_features(ldaps_grid, LDAPS_UV_PAIRS, target)
    gfs_group_grid = add_target_wind_physics_features(gfs_grid, GFS_UV_PAIRS, target)
    ldaps_group_grid = add_group_speed_regime_flags(ldaps_group_grid, target, LDAPS_SPEED_COLS)
    gfs_group_grid = add_group_speed_regime_flags(gfs_group_grid, target, GFS_SPEED_COLS)

    group_frames = [calendar_features]

    if INCLUDE_LDAPS_ALL_GRID_AGGREGATIONS:
        group_frames.append(aggregate_all_grids(ldaps_group_grid, "ldaps"))
    if INCLUDE_GFS_ALL_GRID_AGGREGATIONS:
        group_frames.append(aggregate_all_grids(gfs_group_grid, "gfs"))

    group_frames.extend([
        raw_grid_wide(gfs_group_grid, "gfs", GFS_RAW_GRID_IDS),
        raw_grid_wide(ldaps_group_grid, "ldaps", raw_ids),
    ])

    for subset_name, grid_ids in plan["subsets"].items():
        print(target, subset_name, "basic aggs:", grid_ids)
        group_frames.append(aggregate_subset(ldaps_group_grid, "ldaps", subset_name, grid_ids))

    features = merge_feature_frames(group_frames)
    features = features.sort_values("forecast_kst_dtm").reset_index(drop=True)

    lag_base_cols = choose_lag_roll_cols(features, subset_names)
    lag_excluded_cols = getattr(choose_lag_roll_cols, "last_excluded_cols", [])
    print(target, "lag/rolling base cols:", len(lag_base_cols))
    print(target, "excluded lag/rolling base cols by max limit:", len(lag_excluded_cols))
    features = add_lag_rolling_features(features, lag_base_cols)

    train_df = labels.rename(columns={"kst_dtm": "forecast_kst_dtm"}).merge(features, on="forecast_kst_dtm", how="left")
    test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(features, on="forecast_kst_dtm", how="left")

    if len(test_df) != len(sample_submission):
        numeric_cols = [
            c for c in test_df.columns
            if c not in ["forecast_id", "forecast_kst_dtm"] and pd.api.types.is_numeric_dtype(test_df[c])
        ]
        collapsed = test_df.groupby(["forecast_id", "forecast_kst_dtm"], as_index=False)[numeric_cols].mean(numeric_only=True)
        test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
            collapsed,
            on=["forecast_id", "forecast_kst_dtm"],
            how="left",
        )
    assert len(test_df) == len(sample_submission)

    non_feature_cols = {"forecast_id", "forecast_kst_dtm", *TARGET_COLS}
    datetime_cols = [c for c in train_df.columns if str(train_df[c].dtype).startswith("datetime")]
    non_feature_cols.update(datetime_cols)
    feature_cols = [
        c for c in train_df.columns
        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    combined = pd.concat([train_df[feature_cols], test_df[feature_cols]], ignore_index=True)
    keep_cols = [c for c in feature_cols if combined[c].notna().sum() > 0 and combined[c].nunique(dropna=True) > 1]
    del combined
    gc.collect()

    group_feature_sets[target] = {
        "features": features,
        "train_df": train_df,
        "test_df": test_df,
        "feature_cols": keep_cols,
        "lag_base_cols": lag_base_cols,
        "lag_excluded_cols": lag_excluded_cols,
    }

    pd.DataFrame({"feature": keep_cols}).to_csv(
        FEATURE_OUTPUT_DIR / f"{target}_feature_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame({"feature": lag_base_cols}).to_csv(
        FEATURE_OUTPUT_DIR / f"{target}_lag_rolling_base_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame({"feature": lag_excluded_cols}).to_csv(
        FEATURE_OUTPUT_DIR / f"{target}_lag_rolling_excluded_by_max_features.csv",
        index=False,
        encoding="utf-8-sig",
    )
    feature_inventory_rows.append({
        "target": target,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "feature_count": len(keep_cols),
        "lag_base_count": len(lag_base_cols),
        "lag_excluded_by_max_features_count": len(lag_excluded_cols),
        "max_lag_roll_base_cols": globals().get("MAX_LAG_ROLL_BASE_COLS", None),
        "saved_lag_excluded_list": str(FEATURE_OUTPUT_DIR / f"{target}_lag_rolling_excluded_by_max_features.csv"),
        "ldaps_aggregation_combination_sizes": json.dumps(LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES),
        "raw_ldaps_grid_ids": json.dumps(raw_ids),
        "subset_names": json.dumps(subset_names),
        "count_regime_flags": COUNT_REGIME_FLAGS_IN_AGGREGATIONS,
        "count_precip_flags": COUNT_PRECIP_FLAGS_IN_AGGREGATIONS,
        "include_ldaps_all_grid_aggregations": INCLUDE_LDAPS_ALL_GRID_AGGREGATIONS,
        "include_gfs_all_grid_aggregations": INCLUDE_GFS_ALL_GRID_AGGREGATIONS,
        "weather_lags": json.dumps(WEATHER_LAGS),
        "weather_roll_windows": json.dumps(WEATHER_ROLL_WINDOWS),
    })
    print(target, "train/test/features:", train_df.shape, test_df.shape, len(keep_cols))

feature_inventory = pd.DataFrame(feature_inventory_rows)
feature_inventory.to_csv(FEATURE_INVENTORY_PATH, index=False, encoding="utf-8-sig")
display(feature_inventory)
print("Feature build elapsed_sec:", round(time.time() - started, 1))
print("Saved feature inventory:", FEATURE_INVENTORY_PATH)

## 8. Feature Output Preview

Print a compact summary of the generated feature frames and show representative feature names by group and category.

In [ ]:
def classify_feature_name(feature: str) -> str:
    if "_lag" in feature:
        return "lag"
    if "_roll" in feature:
        return "rolling"
    if "below_cutin" in feature or "cutin_to_rated" in feature or "rated_to_cutout" in feature:
        return "wind_regime"
    if "power_proxy" in feature:
        return "wind_power_proxy"
    if "precip_" in feature:
        return "precipitation_bin"
    if "dry_air_density" in feature or "humid_air_density" in feature:
        return "air_density"
    if feature.startswith("ldaps_all_"):
        return "ldaps_all_grid_agg"
    parts = feature.split("_")
    if feature.startswith("ldaps_g") and len(parts) > 1 and len(parts[1]) == 3 and parts[1][1:].isdigit():
        return "ldaps_raw_grid"
    if feature.startswith("ldaps_"):
        return "ldaps_subset_agg"
    if feature.startswith("gfs_all_"):
        return "gfs_all_grid_agg"
    if feature.startswith("gfs_g") and len(parts) > 1 and len(parts[1]) == 3 and parts[1][1:].isdigit():
        return "gfs_raw_grid"
    if feature in {
        "hour_sin", "hour_cos",
        "day_sin", "day_cos",
        "month_sin", "month_cos",
        "season_sin", "season_cos",
    }:
        return "calendar"
    return "other"


if "group_feature_sets" not in globals():
    raise RuntimeError("Run section 7 before running this preview cell.")

feature_inventory_path = globals().get("FEATURE_INVENTORY_PATH", FEATURE_OUTPUT_DIR / "feature_inventory.csv")
uv_component_cols = {col for u_col, v_col, _ in [*LDAPS_UV_PAIRS, *GFS_UV_PAIRS] for col in (u_col, v_col)}


def is_raw_grid_feature(feature: str) -> bool:
    parts = feature.split("_")
    return len(parts) > 1 and parts[0] in {"ldaps", "gfs"} and len(parts[1]) == 3 and parts[1].startswith("g") and parts[1][1:].isdigit()


def is_subset_grid_feature(feature: str) -> bool:
    return feature.startswith(("ldaps_g1_s_", "ldaps_g2_s_", "ldaps_g3_s_"))


def is_all_grid_feature(feature: str) -> bool:
    return feature.startswith(("ldaps_all_", "gfs_all_"))


def is_uv_component_feature(feature: str) -> bool:
    return any(uv_col in feature for uv_col in uv_component_cols)

summary_rows = []
category_frames = []

for target, fs in group_feature_sets.items():
    feature_cols = fs["feature_cols"]
    train_df = fs["train_df"]
    test_df = fs["test_df"]

    rolling_feature_count = sum("_roll" in c for c in feature_cols)
    lag_feature_count = sum("_lag" in c for c in feature_cols)
    raw_grid_feature_count = sum(is_raw_grid_feature(c) for c in feature_cols)
    subset_grid_feature_count = sum(is_subset_grid_feature(c) for c in feature_cols)
    all_grid_feature_count = sum(is_all_grid_feature(c) for c in feature_cols)
    grid_feature_count = raw_grid_feature_count + subset_grid_feature_count + all_grid_feature_count
    uv_component_feature_count = sum(is_uv_component_feature(c) for c in feature_cols)
    precip_feature_count = sum("precip_" in c for c in feature_cols)

    summary_rows.append({
        "target": target,
        "train_shape": str(train_df.shape),
        "test_shape": str(test_df.shape),
        "feature_count": len(feature_cols),
        "lag_base_count": len(fs["lag_base_cols"]),
        "lag_excluded_by_max_features_count": len(fs.get("lag_excluded_cols", [])),
        "lag_feature_count": lag_feature_count,
        "rolling_feature_count": rolling_feature_count,
        "all_grid_feature_count": all_grid_feature_count,
        "raw_grid_feature_count": raw_grid_feature_count,
        "subset_grid_feature_count": subset_grid_feature_count,
        "grid_feature_count": grid_feature_count,
        "uv_component_feature_count": uv_component_feature_count,
        "precip_feature_count": precip_feature_count,
        "saved_feature_list": str(FEATURE_OUTPUT_DIR / f"{target}_feature_columns.csv"),
    })

    categories = pd.Series([classify_feature_name(c) for c in feature_cols], name="category")
    cat_count = categories.value_counts().rename_axis("category").reset_index(name="n_features")
    cat_count.insert(0, "target", target)
    category_frames.append(cat_count)

summary_df = pd.DataFrame(summary_rows)
category_summary = pd.concat(category_frames, ignore_index=True)

display(summary_df)
display(category_summary.sort_values(["target", "n_features"], ascending=[True, False]))

print("Feature inventory CSV:", feature_inventory_path)
print("Feature output directory:", FEATURE_OUTPUT_DIR)